Creating a AI chatbot that will be a Product Pitch Generator BOT
-a user will describe a product they are planning to develop and the bot will then generate a sales pitch, generate a marketing image, generate a audio
narration of the pitch and also use a tool to create a short marketing tagline.

In [ ]:
#import required libraries
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

import base64
from io import BytesIO
from PIL import Image
import gradio as gr

In [ ]:
# load and initialize the api keys  from the dotenv file
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [3]:
# define the system prompt
system_prompt= """You are a intelligent and creative Pitch Generator Chatbot that incorporates humor into the pitch. Your role is to help
users turn simple product ideas into engaging, persuavive and memorable sales pitches. Your pitches should be clear, energetic, and imaginative while remaining
easy to understand.

Tone and style guidelines:
-be intelligent and creative
-incorporate light humor where appropriate
-keep pitches concise and engaging
-focus on the products main benefit and why customers would care
-use vivid language that helps people imagine using the products

Structure your pitches like this:
-1)Strong hook that grabs attention
-2) a brief explanation of the product
-3) a clear benefit for the customer
-4) a memorable or humorous closing line

Avoid overly long responses. Keep the pitch impactful and easy to read.

Examples:
Example 1:
USer : 'A smart fridge that tells you when food expires'
Pitch : 'Meet the fridge thats smarter than your grocery list. This smart refrigrator  tracks your food and reminds you before anything
expires - so your spinach stops living a secret second life in the back corner. LEss waste, fewer suprise science experiments and a kitchen that 
finally feels under control

"""

System_prompt_tagline = """
You are creative and intelligent chatbot that generates a short marketing tagline for a product. The tagline should be no more than 8 words.
"""


In [4]:
# creating a function that will create a short catchy tagline using the product description
def generate_tagline(productDescription):
    print(f"TAGLINE TOOL CALLED: Creating a short catchy tagline for {productDescription}", flush=True)
    messages = [{"role": "system", "content": System_prompt_tagline},{"role": "user", "content": productDescription}] 
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    print(f"Here is the catchy tagline {response.choices[0].message.content}")
    return response.choices[0].message.content
    
    

In [5]:
# creating a function that will generate an image using the product description
def create_image(productDescription):
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=f"Create an image for {productDescription}, showing potential customers the new product that {productDescription}, in a vibrant , real life style, that catches attention and is marketing gold",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))



In [11]:
#enabling the chat bot to talk
def chatbotVoice(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="coral",    
      input=message
    )
    return response.content

In [ ]:
generate_tagline("pen that tells you when your ink is about to finish")
create_image("pen that tells you when your ink is about to finish")

In [6]:
#creating a tool json definition for the generate_tagline func
tagline_function ={
    "name": "generate_tagline",
    "description": "Creates a short catchy tagline for a product",
    "parameters": {
        "type": "object",
        "properties": {
            "productDescription": {
                "type": "string",
                "description": "Describes the product and what it does",
            },
        },
        "required": ["productDescription"],
        "additionalProperties": False
    }
}

In [ ]:
# #creating a tool json definition for the create image func
# image_function ={
#     "name": "create_image",
#     "description": "Creates a image of the product",
#     "parameters": {
#         "type": "object",
#         "properties": {
#             "product_Description": {
#                 "type": "string",
#                 "description": "Describes the product",
#             },
#         },
#         "required": ["product_Description"],
#         "additionalProperties": False
#     }
# }

In [7]:
tools = [{"type": "function", "function" : tagline_function}]
tools

[{'type': 'function',
  'function': {'name': 'generate_tagline',
   'description': 'Creates a short catchy tagline for a product',
   'parameters': {'type': 'object',
    'properties': {'productDescription': {'type': 'string',
      'description': 'Describes the product and what it does'}},
    'required': ['productDescription'],
    'additionalProperties': False}}}]

In [8]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    image_response = []
    image = None
    

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        tagline_responses,image_response= handle_tool_calls(message)
        messages.append(message)
        messages.extend(tagline_responses)
        print(messages)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    speak = chatbotVoice(reply) 

    if image_response:
        image = create_image(image_response[0])
    
    return history, image,speak


In [9]:
#here we are handling the tool call
def handle_tool_calls(message):
    tagline_response =[]
    image_response =[]
    tool_call = message.tool_calls[0]
    for tool_call in message.tool_calls:
        if tool_call.function.name == "generate_tagline":
            arguments = json.loads(tool_call.function.arguments)
            product_description = arguments.get('productDescription')
            image_response.append(product_description)
            tagline = generate_tagline(product_description)
            tagline_response.append ({
                "role": "tool",
                "content": tagline,
                "tool_call_id": tool_call.id
            })
        
    return tagline_response,image_response



    # elif tool_call.function.name == "create_image" :
    #         arguments = json.loads(tool_call.function.arguments)
    #         product_description = arguments.get('product_Description')
    #         adimage = create_image(product_description)
    #         image_response.append({
    #             "role": "tool",
    #             "content": adimage,
    #             "tool_call_id": tool_call.id
    #         })

In [12]:
#callbacks
#  puts the message written in a chatbox for cleaner look



def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition
#defininign where image , chatbot and audiobox will be located on the screen
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=300, type="messages")
        image_output = gr.Image(height=300, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Pitch Generator Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, image_output, audio_output]
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


TAGLINE TOOL CALLED: Creating a short catchy tagline for A television that runs without electricity
Here is the catchy tagline Watch Anywhere, Powered by Pure Innovation.
TAGLINE TOOL CALLED: Creating a short catchy tagline for An image advert for a television that operates without any electricity, showing people enjoying TV outdoors in nature during a blackout
Here is the catchy tagline Watch Anywhere, Even When Power’s Out!
[{'role': 'system', 'content': "You are a intelligent and creative Pitch Generator Chatbot that incorporates humor into the pitch. Your role is to help\nusers turn simple product ideas into engaging, persuavive and memorable sales pitches. Your pitches should be clear, energetic, and imaginative while remaining\neasy to understand.\n\nTone and style guidelines:\n-be intelligent and creative\n-incorporate light humor where appropriate\n-keep pitches concise and engaging\n-focus on the products main benefit and why customers would care\n-use vivid language that help